<a href="https://colab.research.google.com/github/Abhi926572/Quantum-Computing-2303A51656/blob/main/PROBLEM_32.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pennylane scikit-learn

In [ ]:
from __future__ import annotations

import argparse
import json
import math
import random
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, Tuple

import numpy as np
import pandas as pd
import pennylane as qml
import torch
from pennylane import numpy as pnp
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset


# ----------------------------- Reproducibility -----------------------------

def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# ------------------------------- Datasets ----------------------------------

def generate_noisy_sine(
    length: int = 2400, noise_std: float = 0.15, seed: int = 42
) -> Tuple[np.ndarray, np.ndarray]:
    """Return input features [noisy sine] and clean-sine prediction target."""
    rng = np.random.default_rng(seed)
    t = np.arange(length, dtype=np.float32)
    clean = (
        np.sin(2.0 * np.pi * t / 50.0)
        + 0.35 * np.sin(2.0 * np.pi * t / 17.0)
    ).astype(np.float32)
    noisy = (clean + rng.normal(0.0, noise_std, length)).astype(np.float32)
    x = noisy.reshape(-1, 1)
    y = clean.reshape(-1, 1)
    return x, y


def generate_narma10(
    length: int = 2600, seed: int = 43
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Generate NARMA-10 data.
    Input features contain the driving signal u(t) and generated output y(t).
    Target is the future NARMA output y(t).
    """
    rng = np.random.default_rng(seed)
    u = rng.uniform(0.0, 0.5, length).astype(np.float32)
    y = np.zeros(length, dtype=np.float32)

    # Standard NARMA-10 recurrence; the first 10 outputs are initialized to 0.
    for t in range(10, length - 1):
        y[t + 1] = (
            0.3 * y[t]
            + 0.05 * y[t] * np.sum(y[t - 9 : t + 1])
            + 1.5 * u[t - 9] * u[t]
            + 0.1
        )

    x = np.column_stack((u, y)).astype(np.float32)
    target = y.reshape(-1, 1)
    return x, target


# -------------------------- Chronological windows --------------------------

@dataclass
class DataBundle:
    train_loader: DataLoader
    val_x: torch.Tensor
    val_y: torch.Tensor
    test_x: torch.Tensor
    test_y: torch.Tensor
    target_scaler: StandardScaler
    input_size: int
    split_sizes: Dict[str, int]


def make_windows(
    x: np.ndarray, y: np.ndarray, lookback: int, horizon: int
) -> Tuple[np.ndarray, np.ndarray]:
    xs, ys = [], []
    # Each output is the next `horizon` target values after the input window.
    for end in range(lookback, len(x) - horizon + 1):
        xs.append(x[end - lookback : end])
        ys.append(y[end : end + horizon, 0])
    if not xs:
        raise ValueError("Not enough observations for the selected lookback/horizon.")
    return np.asarray(xs, dtype=np.float32), np.asarray(ys, dtype=np.float32)


def prepare_data(
    x: np.ndarray,
    y: np.ndarray,
    lookback: int,
    horizon: int,
    batch_size: int,
) -> DataBundle:
    n = len(x)
    train_end = int(n * 0.70)
    val_end = int(n * 0.85)

    # Raw observations are partitioned first. Windows are then made separately.
    raw = {
        "train": (x[:train_end], y[:train_end]),
        "val": (x[train_end:val_end], y[train_end:val_end]),
        "test": (x[val_end:], y[val_end:]),
    }

    # Fit scalers only on the training segment.
    x_scaler = StandardScaler().fit(raw["train"][0])
    y_scaler = StandardScaler().fit(raw["train"][1])

    windows = {}
    for split, (sx, sy) in raw.items():
        sx_scaled = x_scaler.transform(sx).astype(np.float32)
        sy_scaled = y_scaler.transform(sy).astype(np.float32)
        windows[split] = make_windows(sx_scaled, sy_scaled, lookback, horizon)

    train_x, train_y = windows["train"]
    val_x, val_y = windows["val"]
    test_x, test_y = windows["test"]

    train_ds = TensorDataset(torch.from_numpy(train_x), torch.from_numpy(train_y))
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

    return DataBundle(
        train_loader=train_loader,
        val_x=torch.from_numpy(val_x),
        val_y=torch.from_numpy(val_y),
        test_x=torch.from_numpy(test_x),
        test_y=torch.from_numpy(test_y),
        target_scaler=y_scaler,
        input_size=x.shape[1],
        split_sizes={k: len(v[0]) for k, v in raw.items()},
    )


# ------------------------------- QLSTM -------------------------------------

class QuantumLayer(nn.Module):
    """Variational quantum layer returning one Pauli-Z expectation per qubit."""

    def __init__(self, n_qubits: int = 4, n_layers: int = 1):
        super().__init__()
        self.n_qubits = n_qubits
        self.n_layers = n_layers

        # Remove shots to enable analytic device simulation required for backpropagation
        dev = qml.device("default.qubit", wires=n_qubits)

        @qml.qnode(dev, interface="torch", diff_method="backprop")
        def circuit(inputs, weights):
            qml.AngleEmbedding(inputs, wires=range(n_qubits), rotation="Y")
            qml.StronglyEntanglingLayers(weights, wires=range(n_qubits))
            return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

        weight_shapes = {"weights": (n_layers, n_qubits, 3)}
        self.layer = qml.qnn.TorchLayer(circuit, weight_shapes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.layer(x)


class QLSTMCell(nn.Module):
    def __init__(
        self, input_size: int, hidden_size: int, n_qubits: int, q_layers: int
    ):
        super().__init__()
        self.hidden_size = hidden_size
        self.to_quantum = nn.Linear(input_size + hidden_size, n_qubits)
        self.quantum = QuantumLayer(n_qubits, q_layers)
        self.to_gates = nn.Linear(n_qubits, 4 * hidden_size)

    def forward(self, x_t, state):
        h, c = state
        combined = torch.cat((x_t, h), dim=1)
        q_in = torch.tanh(self.to_quantum(combined)) * math.pi
        q_out = self.quantum(q_in)
        i, f, g, o = self.to_gates(q_out).chunk(4, dim=1)

        i = torch.sigmoid(i)
        f = torch.sigmoid(f)
        g = torch.tanh(g)
        o = torch.sigmoid(o)

        c_next = f * c + i * g
        h_next = o * torch.tanh(c_next)
        return h_next, c_next


class QuantumLSTM(nn.Module):
    def __init__(
        self,
        input_size: int,
        hidden_size: int,
        horizon: int,
        n_qubits: int = 4,
        q_layers: int = 1,
    ):
        super().__init__()
        self.hidden_size = hidden_size
        self.cell = QLSTMCell(input_size, hidden_size, n_qubits, q_layers)
        self.output = nn.Linear(hidden_size, horizon)

    def forward(self, x):
        batch_size = x.shape[0]
        h = x.new_zeros(batch_size, self.hidden_size)
        c = x.new_zeros(batch_size, self.hidden_size)

        for t in range(x.shape[1]):
            h, c = self.cell(x[:, t, :], (h, c))
        return self.output(h)


# ----------------------------- Train/evaluate ------------------------------

def inverse_targets(values: np.ndarray, scaler: StandardScaler) -> np.ndarray:
    return scaler.inverse_transform(values.reshape(-1, 1)).reshape(values.shape)


def evaluate(
    model: nn.Module,
    x: torch.Tensor,
    y: torch.Tensor,
    scaler: StandardScaler,
    device: torch.device,
) -> Tuple[Dict, np.ndarray, np.ndarray]:
    model.eval()
    with torch.no_grad():
        pred_scaled = model(x.to(device)).cpu().numpy()
    true_scaled = y.numpy()

    pred = inverse_targets(pred_scaled, scaler)
    true = inverse_targets(true_scaled, scaler)

    residual = pred - true
    metrics = {
        "MAE": float(mean_absolute_error(true.reshape(-1), pred.reshape(-1))),
        "RMSE": float(np.sqrt(mean_squared_error(true.reshape(-1), pred.reshape(-1)))),
        "R2": float(r2_score(true.reshape(-1), pred.reshape(-1))),
        "residual_bias": float(np.mean(residual)),
        "horizon_MAE": [
            float(mean_absolute_error(true[:, h], pred[:, h]))
            for h in range(true.shape[1])
        ],
        "horizon_RMSE": [
            float(np.sqrt(mean_squared_error(true[:, h], pred[:, h])))
            for h in range(true.shape[1])
        ],
    }
    return metrics, true, pred


def train_model(
    model: nn.Module,
    data: DataBundle,
    epochs: int,
    learning_rate: float,
    device: torch.device,
) -> Tuple[list, float]:
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    loss_fn = nn.MSELoss()
    history = []
    start = time.perf_counter()

    for epoch in range(epochs):
        model.train()
        batch_losses = []
        for xb, yb in data.train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            pred = model(xb)
            loss = loss_fn(pred, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            batch_losses.append(float(loss.detach().cpu()))

        model.eval()
        with torch.no_grad():
            val_pred = model(data.val_x.to(device))
            val_loss = float(loss_fn(val_pred, data.val_y.to(device)).cpu())

        history.append({
            "epoch": epoch + 1,
            "train_loss": float(np.mean(batch_losses)),
            "validation_loss": val_loss,
        })
        print(
            f"Epoch {epoch + 1:02d}/{epochs} | "
            f"train loss={history[-1]['train_loss']:.6f} | "
            f"validation loss={val_loss:.6f}"
        )

    training_seconds = time.perf_counter() - start
    return history, training_seconds


# --------------------------------- Runner -----------------------------------

def run_experiment(args, dataset_name: str, x: np.ndarray, y: np.ndarray) -> None:
    print(f"\n{'=' * 18} {dataset_name.upper()} {'=' * 18}")
    data = prepare_data(x, y, args.lookback, args.horizon, args.batch_size)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = QuantumLSTM(
        input_size=data.input_size,
        hidden_size=args.hidden_size,
        horizon=args.horizon,
        n_qubits=args.qubits,
        q_layers=args.circuit_layers,
    ).to(device)

    history, training_seconds = train_model(
        model, data, args.epochs, args.learning_rate, device
    )
    metrics, _, _ = evaluate(
        model, data.test_x, data.test_y, data.target_scaler, device
    )

    # Resource estimates: each sample/time-step passes through one quantum layer.
    train_windows = len(data.train_loader.dataset)
    circuit_calls_per_epoch = train_windows * args.lookback
    total_circuit_calls = circuit_calls_per_epoch * args.epochs
    # Parameter-shift gradients require additional circuit executions.
    estimated_forward_shots = total_circuit_calls * args.shots

    quantum_cost = {
        "qubits": args.qubits,
        "modes": args.qubits,
        "circuit_layers": args.circuit_layers,
        "estimated_circuit_depth": 1 + 2 * args.circuit_layers,
        "shots_per_circuit": args.shots,
        "estimated_circuit_evaluations_forward_only": total_circuit_calls,
        "estimated_forward_shots": estimated_forward_shots,
        "training_time_seconds": training_seconds,
        "inference_time_seconds": None,
    }

    model.eval()
    inference_start = time.perf_counter()
    with torch.no_grad():
        _ = model(data.test_x[: min(16, len(data.test_x))].to(device))
    inference_elapsed = time.perf_counter() - inference_start
    quantum_cost["inference_time_seconds"] = inference_elapsed

    result = {
        "dataset": dataset_name,
        "configuration": {
            "lookback": args.lookback,
            "forecast_horizon": args.horizon,
            "hidden_size": args.hidden_size,
            "epochs": args.epochs,
            "batch_size": args.batch_size,
            "learning_rate": args.learning_rate,
            "device": str(device),
            "split_raw_observations": data.split_sizes,
        },
        "test_metrics": metrics,
        "training_stability": {
            "epoch_history": history,
            "final_train_loss": history[-1]["train_loss"],
            "final_validation_loss": history[-1]["validation_loss"],
            "train_loss_std_last_5_epochs": float(
                np.std([r["train_loss"] for r in history[-5:]])
            ),
        },
        "quantum_cost": quantum_cost,
    }

    output_dir = Path(args.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    safe_name = dataset_name.lower().replace("-", "_")
    with (output_dir / f"{safe_name}_results.json").open("w", encoding="utf-8") as f:
        json.dump(result, f, indent=2)
    pd.DataFrame(history).to_csv(
        output_dir / f"{safe_name}_training_history.csv", index=False
    )

    print("\nTest metrics:")
    for key, value in metrics.items():
        print(f"  {key}: {value}")
    print("\nQuantum cost:")
    for key, value in quantum_cost.items():
        print(f"  {key}: {value}")
    print(f"\nSaved results in: {output_dir.resolve()}")


def parse_args():
    parser = argparse.ArgumentParser(description="Quantum LSTM temporal prediction")
    parser.add_argument("--dataset", choices=["sine", "narma10", "both"], default="both")
    parser.add_argument("--epochs", type=int, default=10)
    parser.add_argument("--lookback", type=int, default=20)
    parser.add_argument("--horizon", type=int, default=5)
    parser.add_argument("--hidden-size", type=int, default=8)
    parser.add_argument("--qubits", type=int, default=4)
    parser.add_argument("--circuit-layers", type=int, default=1)
    parser.add_argument("--shots", type=int, default=1000)
    parser.add_argument("--batch-size", type=int, default=16)
    parser.add_argument("--learning-rate", type=float, default=0.005)
    parser.add_argument("--output-dir", type=str, default="qlstm_outputs")
    parser.add_argument("--seed", type=int, default=42)
    return parser.parse_known_args()[0]


def main():
    args = parse_args()
    if args.epochs < 1 or args.lookback < 1 or args.horizon < 1:
        raise ValueError("epochs, lookback, and horizon must be positive.")
    if args.qubits < 2:
        raise ValueError("Use at least 2 qubits.")
    set_seed(args.seed)

    if args.dataset in ("sine", "both"):
        x, y = generate_noisy_sine(seed=args.seed)
        run_experiment(args, "D19_noisy_sine", x, y)

    if args.dataset in ("narma10", "both"):
        x, y = generate_narma10(seed=args.seed + 1)
        run_experiment(args, "D20_narma10", x, y)


if __name__ == "__main__":
    main()


================== D19_NOISY_SINE ==================
Epoch 01/10 | train loss=0.596475 | validation loss=0.245281
Epoch 02/10 | train loss=0.214495 | validation loss=0.213683
Epoch 03/10 | train loss=0.142057 | validation loss=0.150757
Epoch 04/10 | train loss=0.121900 | validation loss=0.131806
Epoch 05/10 | train loss=0.111843 | validation loss=0.125837
Epoch 06/10 | train loss=0.098863 | validation loss=0.101720
Epoch 07/10 | train loss=0.093034 | validation loss=0.082938
Epoch 08/10 | train loss=0.085024 | validation loss=0.065620
Epoch 09/10 | train loss=0.075713 | validation loss=0.057153
Epoch 10/10 | train loss=0.071863 | validation loss=0.052495

Test metrics:
  MAE: 0.1513056457042694
  RMSE: 0.1882234959644485
  R2: 0.9374809861183167
  residual_bias: 0.021610813215374947
  horizon_MAE: [0.12825153768062592, 0.1467638462781906, 0.16044528782367706, 0.15802757441997528, 0.16303999722003937]
  horizon_RMSE: [0.1596286795336944, 0.1812220025636527, 0.19632897042495853, 0.19880